In [15]:
import pandas as pd
from tqdm import tqdm
def load_df(fp: str) -> pd.DataFrame:
    with open(fp, 'r', encoding='utf-8') as f:
        # Read header and split into column names
        header = f.readline().strip().split('\t')
        
        rows = []
        for i, line in tqdm(enumerate(f, start=2)):  # start=2 for correct line numbering
            line = line.strip()
            if not line:
                print(f"Skipping blank line at {i}")
                continue
            
            values = line.split('\t')
            if len(values) > 4:
                # Merge values[2:-1] (from third to second last) into one string
                merged = '\t'.join(values[2:-1])
                # Reconstruct the list with exactly 4 elements
                values = [values[0], values[1], merged, values[-1]]
                
            if len(values) != len(header):
                print(f"Malformed row at line {i}: Expected {len(header)} fields, got {len(values)}")
                continue

            rows.append(values)
    
    # Create DataFrame from list of rows
    df = pd.DataFrame(rows, columns=header)

    # Optionally enforce column types
    df = df.astype({
        "qid": str,
        "pid": str,
        "answer": str,
        "target": str
    })  
    # Must read qid and pid as string not as int
    # dtype_spec = {"qid": str, "pid": str, "answer": str, "target": str}
    # df = pd.read_csv(fp, delimiter="\t", dtype=dtype_spec)
    return df

In [44]:
import os
dataset_name = "trec-2022"
task = "title_generation"
query_topic = "10"
retriever_name = "bm25"
MODEL_NAME = "llama31-8b"
INF_RESULTS_DIR_PATH = os.path.join(
    "./inference_results",
    dataset_name,
    task,
    query_topic,
    retriever_name,
    MODEL_NAME,
)

In [45]:
df = load_df(os.path.join(INF_RESULTS_DIR_PATH, "5_output_augment.log"))
skipped_qid = set()
for eachrow in df.itertuples(index=False):
    qid = eachrow.qid
    answer = eachrow.answer
    if answer.strip() == "":
        skipped_qid.add(qid)

df = load_df(os.path.join(INF_RESULTS_DIR_PATH, "5_output_baseline.log"))
for eachrow in df.itertuples(index=False):
    qid = eachrow.qid
    answer = eachrow.answer
    if answer.strip() == "":
        skipped_qid.add(qid)

df = load_df(os.path.join(INF_RESULTS_DIR_PATH, "5_output_rag.log"))
for eachrow in df.itertuples(index=False):
    qid = eachrow.qid
    answer = eachrow.answer
    if answer.strip() == "":
        skipped_qid.add(qid)

1400it [00:00, 591461.08it/s]
140it [00:00, 449963.65it/s]
140it [00:00, 541699.78it/s]


In [46]:
len(skipped_qid)

0